In [2]:
pip install numpy pandas matplotlib statsmodels scikit-learn pmdarima

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 25.5 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 44.4 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 47.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 46.4 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 45.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 24.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 42.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 46.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 42.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 52.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 45.5 MB/s  0:00:006m0:00:01
   ━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  3/17 [numpy]  WARNING: The scripts f2py and numpy-config are installed in '/usr/local/pyth

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Load your data
loandata = pd.read_csv("Climate Data/BUSLOANS.csv")
unemployment = pd.read_csv("Climate Data/UNEMPLOYMENT.csv")
cpi = pd.read_csv("Climate Data/CPI.csv")

# Convert dates to datetime
loandata['Date'] = pd.to_datetime(loandata['observation_date'])
unemployment['Date'] = pd.to_datetime(unemployment['observation_date'])
cpi['Date'] = pd.to_datetime(cpi['observation_date'])

# Merge datasets
merged_data = pd.merge(loandata, unemployment, on='observation_date', how='inner')
merged_data = pd.merge(merged_data, cpi, on='observation_date', how='inner')

# Calculate loan growth and inflation
merged_data['loan_growth'] = merged_data['BUSLOANS'].pct_change() * 100
merged_data['inflation'] = merged_data['CPIAUCSL'].pct_change() * 100

# Drop NaN values
merged_data = merged_data.dropna()

# Set date as index
merged_data['observation_date'] = pd.to_datetime(merged_data['observation_date'])
merged_data.set_index('observation_date', inplace=True)

loan_growth = merged_data['loan_growth']
unemployment_rate = merged_data['UNRATE']

# Grid search - with better error reporting
best_aic = np.inf
best_params = None
results_list = []

for p, d, q in product(range(0, 5), range(0, 2), range(0, 5)):
    try:
        model = sm.tsa.SARIMAX(
            loan_growth, 
            order=(p, d, q),
            exog=unemployment_rate
        )
        results = model.fit(disp=False)
        results_list.append({
            'order': f'({p},{d},{q})', 
            'AIC': results.aic, 
            'BIC': results.bic
        })
        if results.aic < best_aic:
            best_aic = results.aic
            best_params = (p, d, q)
    except Exception as e:
        print(f"Failed for order ({p},{d},{q}): {str(e)[:50]}")
        continue

# Display results
if results_list:
    results_df = pd.DataFrame(results_list).sort_values('AIC')
    print(results_df.head(10))
    print(f"\nOptimal ARIMA order: {best_params}")
else:
    print("No successful model fits!")

/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored

      order          AIC          BIC
33  (3,0,3)  2203.948480  2242.664292
24  (2,0,4)  2205.154260  2243.870072
34  (3,0,4)  2205.663231  2249.218519
43  (4,0,3)  2205.727294  2249.282582
44  (4,0,4)  2207.677750  2256.072514
12  (1,0,2)  2209.159377  2233.356759
13  (1,0,3)  2209.544916  2238.581775
22  (2,0,2)  2209.735159  2238.772018
31  (3,0,1)  2209.983391  2239.020250
32  (3,0,2)  2210.844210  2244.720545

Optimal ARIMA order: (3, 0, 3)


/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
